# Test what telephone inbound evidence adds

**Inherited contract:** the reproduced national benchmark and the nested 3,020-practice CBT inbound cohort.

**Purpose:** calculate the matched comparison between the 14-feature control and 17-feature CBT inbound model on the same 3,020 practices.

The additional variables are inbound calls per 1,000 registered patient-months, mean absolute monthly call-rate change and call-rate range. The full-data comparison is recomputed here. Closed resampling evidence is inspected only after its authority checksum is validated.

In [1]:
from pathlib import Path
import sys

ROOT = Path.cwd()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'src'))
CONFIG_PATH = ROOT / 'configs' / 'reference_apr2025_mar2026.json'
from gpap2.config import load_config
REFERENCE_CONFIG = load_config(CONFIG_PATH)
AUTHORITY_MANIFEST = REFERENCE_CONFIG.resolve(REFERENCE_CONFIG.authority_checksum_file)
import pandas as pd
pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 120)
pd.set_option('display.precision', 6)


## Method contract

This stage inherits the fixed national feature values for the nested CBT-observed cohort. It fits the shared 14-feature baseline and 17-feature inbound specification under identical preprocessing and K-Means controls. The displayed values come from the [reference configuration](../configs/reference_apr2025_mar2026.json), [comparison implementation](../src/gpap2/analysis.py), [preprocessing implementation](../src/gpap2/preprocessing.py) and [reporting helpers](../src/gpap2/notebook_reporting.py).

In [2]:
from gpap2.analysis import compare_inbound_models
from gpap2.config import load_config
from gpap2.io import read_contract_csv, validate_authority_file
from gpap2.notebook_reporting import (
    build_comparison_contract_table,
    build_feature_contract_table,
    build_model_contract_table,
    build_transformation_contract_table,
)
from gpap2.validation import validate_cohort_relationships

config = load_config(CONFIG_PATH)
source = config.resolve(config.input_directory) / config.specification('cbt_inbound_17').source_file
inbound = read_contract_csv(source)
comparison = compare_inbound_models(inbound, config)
cohort_checks = validate_cohort_relationships(config.resolve(config.input_directory), config)
cohort_checks.loc[cohort_checks['test'].str.startswith('CBT inbound')]

,test,expected,observed,passed
0,CBT inbound practices are nested in national,0,0,True
2,CBT inbound inherits all national feature valu...,0,0,True


In [3]:
feature_contract = build_feature_contract_table(config, 'cbt_inbound_17')
feature_contract

,feature_order,feature,transformation,included_in_model
0,1,ocs_submissions_per_1000_patient_months,log1p,True
1,2,ocs_clinical_share,unchanged,True
2,3,ocs_administrative_share,unchanged,True
3,4,gpad_appointments_per_1000_patient_months,log1p,True
4,5,gpad_dna_share,unchanged,True
5,6,gpad_face_to_face_share,unchanged,True
6,7,gpad_telephone_share,unchanged,True
7,8,gpad_same_day_share,unchanged,True
8,9,gpad_1_day_share,unchanged,True
9,10,gpad_2_to_7_days_share,unchanged,True


In [4]:
transformation_contract = build_transformation_contract_table(
    comparison.prepared['cbt_inbound_17']
)
transformation_contract

,feature_order,feature,transformation,fitted_median,pre_scaling_iqr,fitted_scale,iqr_gate_passed
0,1,ocs_submissions_per_1000_patient_months,log1p,4.224822,2.089468,2.089468,True
1,2,ocs_clinical_share,unchanged,0.670100,0.255752,0.255752,True
2,3,ocs_administrative_share,unchanged,0.290998,0.226591,0.226591,True
3,4,gpad_appointments_per_1000_patient_months,log1p,6.137541,0.356566,0.356566,True
4,5,gpad_dna_share,unchanged,0.041980,0.028431,0.028431,True
5,6,gpad_face_to_face_share,unchanged,0.652363,0.226515,0.226515,True
6,7,gpad_telephone_share,unchanged,0.231522,0.190592,0.190592,True
7,8,gpad_same_day_share,unchanged,0.420273,0.163251,0.163251,True
8,9,gpad_1_day_share,unchanged,0.071722,0.039152,0.039152,True
9,10,gpad_2_to_7_days_share,unchanged,0.178256,0.082662,0.082662,True


In [5]:
build_model_contract_table(config, comparison.prepared['cbt_inbound_17'])

,setting,value,runtime_source
0,algorithm,K-Means,src/gpap2/models.py
1,clusters (k),3,reference configuration
2,initialisation,k-means++,src/gpap2/models.py
3,n_init,100,reference configuration
4,max_iter,500,reference configuration
5,random_state,2026,reference configuration
6,implementation,lloyd,reference configuration
7,centering,median,reference configuration
8,scaling,interquartile range (IQR),reference configuration
9,label alignment,maximum-agreement Hungarian assignment,src/gpap2/models.py


In [6]:
stability_path = ROOT / 'outputs' / 'tables' / 'robustness_summary.csv'
validate_authority_file(stability_path, AUTHORITY_MANIFEST)
stability = read_contract_csv(stability_path)
stability.loc[stability['robustness_domain'].eq('CBT inbound sensitivity')]

,robustness_domain,cohort_n,result,interpretation,limitation
5,CBT inbound sensitivity,3020,571 practices (18.9%) changed after CBT inboun...,Telephone activity can alter membership in the...,Restricted cohort; not nationally complete.


In [7]:
import numpy as np
from gpap2.analysis import canonical_assignment_sha256

row = comparison.comparisons.iloc[0]
assignment_hash = canonical_assignment_sha256(
    comparison.assignments,
    config.identifier,
    'cbt_inbound_17_aligned_to_national_14',
)
authority_path = ROOT / 'outputs' / 'validation' / 'analytical_regression_results.csv'
validate_authority_file(authority_path, AUTHORITY_MANIFEST)
authority = read_contract_csv(authority_path)
expected = authority.loc[
    authority['reference'].eq(row['reference']) & authority['candidate'].eq(row['candidate'])
].iloc[0]
for metric in ['adjusted_rand_index', 'normalised_mutual_information', 'aligned_agreement']:
    assert np.isclose(row[metric], expected[metric], atol=1e-12, rtol=0)
assert int(row['reassigned_practices']) == int(expected['reassigned_practices'])
assert assignment_hash == expected['canonical_aligned_assignment_sha256']
comparison_contract = build_comparison_contract_table(
    comparison.comparisons,
    comparison.diagnostics,
    {'cbt_inbound_17': assignment_hash},
)
comparison_contract

,reference,candidate,adjusted_rand_index,normalised_mutual_information,aligned_agreement,reassigned_practices,reference_silhouette,candidate_silhouette,canonical_aligned_assignment_sha256
0,national_14,cbt_inbound_17,0.523613,0.468955,0.810927,571,0.126528,0.115252,892AFAF3EC4CEB9D6B1D7DC659580CE3E11C8809ED7D79...


## Decision

Inbound telephone activity is an informative restricted-cohort sensitivity. It changes a material set of assignments but does not replace the national OCS-GPAD profile model or imply national CBT coverage.

**Stage handover:** The inbound result remains a restricted evidence-availability sensitivity and defines the baseline for testing whether telephone-outcome representation changes that sensitivity. It does not replace or rerun the national model.